In [ ]:
"""
Input-Conditioned Weight Initialization — Proof of Concept
Multi-seed averaged curves + grid search + t-tests.

Run on Colab T4:
    !pip install scikit-learn
    !python conditioned_init_experiment.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from itertools import product
from scipy import stats
import tensorflow as tf
import random

# ── Config ────────────────────────────────────────────────────────────────────
SEEDS      = [42, 7, 13, 99, 2025, 1, 8, 21]   # 8 seeds for t-test power
N_STEPS    = 500
BATCH_SIZE = 256
LR         = 1e-3

GRID = {
    #half of number of filters
    'n_clusters' : [16],
    #n_patches  = (filters / 4) × (1 / foreground_density)
    'n_patches'  : [25],
    # .1-.3 works, genuine hyperparam - good to be less restrictive than Kaim since BN can address it
    'scale'      : [.1, .15, .2, .25, .3],
    # size of kernel
    'patch_size' : [3],
}
# ─────────────────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


# ── Data ──────────────────────────────────────────────────────────────────────
print('Loading Fashion-MNIST...')
(x_train, y_train), _ = tf.keras.datasets.fashion_mnist.load_data()

MEAN, STD = 0.2860, 0.3530
x_train = (x_train.astype(np.float32) / 255.0 - MEAN) / STD
x_train = x_train[:, np.newaxis, :, :]

dataset     = torch.utils.data.TensorDataset(
    torch.tensor(x_train),
    torch.tensor(y_train.astype(np.int64))
)
trainloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
print(f'Dataset ready: {len(dataset)} samples\n')


# ── Model ─────────────────────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    def __init__(self, use_bn1=True):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 7 * 7, 256)
        self.fc2   = nn.Linear(256, 10)
        self.bn1   = nn.BatchNorm2d(32) if use_bn1 else nn.Identity()
        self.bn2   = nn.BatchNorm2d(64)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# ── Conditioner ───────────────────────────────────────────────────────────────
def extract_patches(images, patch_size, n_patches):
    B, C, H, W = images.shape
    patches = []
    imgs_np = images.cpu().numpy()
    for b in range(B):
        for _ in range(n_patches):
            i = random.randint(0, H - patch_size)
            j = random.randint(0, W - patch_size)
            patch = imgs_np[b, :, i:i+patch_size, j:j+patch_size]
            patches.append(patch.flatten())
    return np.array(patches)


def distance_weights(patch_size):
    center  = patch_size // 2
    weights = []
    for _ in range(1):
        for i in range(patch_size):
            for j in range(patch_size):
                dist = abs(i - center) + abs(j - center)
                weights.append(1.0 / (1.0 + dist))
    return np.array(weights)


def conditioned_init(model, batch_images, n_clusters, n_patches, scale, patch_size, seed):
    patches = extract_patches(batch_images, patch_size, n_patches)
    patches -= patches.mean(axis=1, keepdims=True)

    kmeans    = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, n_init=3)
    kmeans.fit(patches)
    centroids = kmeans.cluster_centers_.copy()

    dw        = distance_weights(patch_size)
    centroids = centroids * dw[np.newaxis, :]
    centroids = centroids / (np.std(centroids) + 1e-8) * scale

    weight_tensor = torch.tensor(
        centroids.reshape(n_clusters, 1, patch_size, patch_size),
        dtype=torch.float32
    )

    with torch.no_grad():
        target_weight = model.conv1.weight
        out_channels  = target_weight.shape[0]
        if patch_size != 3:
            start         = (patch_size - 3) // 2
            weight_tensor = weight_tensor[:, :, start:start+3, start:start+3].contiguous()
        num_to_copy = min(n_clusters, out_channels)
        target_weight[:num_to_copy].copy_(weight_tensor[:num_to_copy])

    return model


# ── Single run ────────────────────────────────────────────────────────────────
def run(seed, conditioned=False, hp=None, use_bn1=True):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    model     = SimpleCNN(use_bn1=use_bn1).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    losses    = []

    data_iter      = iter(trainloader)
    images, labels = next(data_iter)

    if conditioned and hp is not None:
        model = conditioned_init(model, images, **hp, seed=seed)

    model.eval()
    with torch.no_grad():
        losses.append(criterion(model(images.to(device)), labels.to(device)).item())

    model.train()
    step = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        step += 1
        if step >= N_STEPS:
            break

    return losses


# ── Multi-seed runner ─────────────────────────────────────────────────────────
def run_seeds(conditioned=False, hp=None, use_bn1=True):
    all_losses = [run(s, conditioned=conditioned, hp=hp, use_bn1=use_bn1) for s in SEEDS]
    arr = np.array(all_losses)
    return arr.mean(axis=0), arr.std(axis=0), [r[-1] for r in all_losses]


# ── Plot helper ───────────────────────────────────────────────────────────────
def plot_comparison(baseline_mean, baseline_std, cond_mean, cond_std, title, fname):
    steps = np.arange(len(baseline_mean))
    plt.figure(figsize=(11, 5))
    plt.plot(steps, baseline_mean, label='Kaiming', color='steelblue')
    plt.fill_between(steps, baseline_mean - baseline_std,
                             baseline_mean + baseline_std, alpha=0.2, color='steelblue')
    plt.plot(steps, cond_mean, label='Conditioned', color='tomato')
    plt.fill_between(steps, cond_mean - cond_std,
                             cond_mean + cond_std, alpha=0.2, color='tomato')
    plt.xlabel('Step')
    plt.ylabel('Cross-Entropy Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f'  Saved → {fname}')


# ── Grid search + t-tests for one bn condition ────────────────────────────────
def run_experiment(use_bn1, tag):
    label = 'with_bn1' if use_bn1 else 'no_bn1'

    print(f'\n{"="*60}')
    print(f'  Condition: {label}')
    print(f'{"="*60}')

    print('Running baseline...')
    base_mean, base_std, base_finals = run_seeds(conditioned=False, use_bn1=use_bn1)
    print(f'  Baseline step-{N_STEPS} loss: {base_mean[-1]:.4f} ± {base_std[-1]:.4f}')

    keys   = list(GRID.keys())
    combos = list(product(*GRID.values()))
    print(f'\nGrid search: {len(combos)} combinations × {len(SEEDS)} seeds\n')

    results = []
    for i, combo in enumerate(combos):
        hp = dict(zip(keys, combo))
        cond_mean, cond_std, cond_finals = run_seeds(conditioned=True, hp=hp, use_bn1=use_bn1)
        delta                            = base_mean[-1] - cond_mean[-1]
        t_stat, p_value                  = stats.ttest_rel(base_finals, cond_finals)
        sig = '***' if p_value < 0.01 else '** ' if p_value < 0.05 else '*  ' if p_value < 0.1 else 'ns '
        print(f'[{i+1:>3}/{len(combos)}] delta={delta:+.4f}  p={p_value:.4f} {sig}  {hp}')
        results.append((delta, p_value, sig, hp, cond_mean, cond_std, cond_finals))

    results.sort(key=lambda x: x[0], reverse=True)

    print(f'\n── Rankings ({label}) ────────────────────────────────────────')
    print(f'{"Rank":<5} {"Delta":>8}  {"p-value":>8}  {"sig":>4}  config')
    for rank, (delta, p_value, sig, hp, *_) in enumerate(results, 1):
        print(f'{rank:<5} {delta:>+8.4f}  {p_value:>8.4f}  {sig}  {hp}')

    # Plot best, median, worst
    best_delta,  p, s, best_hp,  best_mean,  best_std,  _  = results[0]
    mid = len(results) // 2
    med_delta,   p, s, med_hp,   med_mean,   med_std,   _  = results[mid]
    worst_delta, p, s, worst_hp, worst_mean, worst_std, _  = results[-1]

    plot_comparison(base_mean, base_std, best_mean,  best_std,
        f'Best ({label})  Δ={best_delta:+.4f}  {best_hp}',   f'best_{label}.png')
    plot_comparison(base_mean, base_std, med_mean,   med_std,
        f'Median ({label})  Δ={med_delta:+.4f}  {med_hp}',   f'median_{label}.png')
    plot_comparison(base_mean, base_std, worst_mean, worst_std,
        f'Worst ({label})  Δ={worst_delta:+.4f}  {worst_hp}', f'worst_{label}.png')

    return results


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    results_with_bn1 = run_experiment(use_bn1=True,  tag='with_bn1')
    #results_no_bn1  = run_experiment(use_bn1=False, tag='no_bn1')


    print('\n── Head-to-head: best config each condition ──────────────────────')
    #best_no_bn1   = results_no_bn1[0]
    best_with_bn1 = results_with_bn1[0]
    #print(f'  no_bn1   best: delta={best_no_bn1[0]:+.4f}  p={best_no_bn1[1]:.4f}  {best_no_bn1[3]}')
    print(f'  with_bn1 best: delta={best_with_bn1[0]:+.4f}  p={best_with_bn1[1]:.4f}  {best_with_bn1[3]}')
    print('\nDone.')

# Input-Conditioned Weight Initialization: Preliminary Findings

## Hypothesis
A zero-training-cost, input-conditioned initialization of the first convolutional layer
reduces initial loss and improves convergence speed compared to standard Kaiming random
initialization.

## Method
Before the first forward pass, k-means is run on mean-centered 3×3 patches sampled from
the first training batch. The resulting centroids are distance-weighted and normalized,
then assigned to half of the first convolutional layer's filters. The remaining filters
retain their Kaiming initialization. No gradient steps are used.

**Conditioner pipeline:**
1. Estimate foreground density `d` via Otsu's method on raw [0,1] pixels
2. Compute `n_patches = (F/4) × (1/d)` where F = number of conv1 filters
3. Sample `n_patches` random 3×3 patches per image from the first training batch
4. Mean-center each patch (encodes texture and edges, not brightness)
5. Run MiniBatchKMeans with `n_clusters = F/2` to find centroids
6. Apply inverse Manhattan distance spatial weighting to each centroid
7. Normalize centroids to target std (`scale`)
8. Write centroids directly into the first `n_clusters` conv1 filter weights

## Experimental Setup
- **Dataset:** Fashion-MNIST (60,000 samples)
- **Architecture:** SimpleCNN with BatchNorm (standard configuration, bn1 included)
- **Training:** 500 steps, Adam optimizer, lr=1e-3, batch size=256
- **Evaluation:** Paired t-test across 8 independent seeds
- **Comparison:** Kaiming uniform initialization (baseline)

## Key Findings

### 1. Effect is statistically significant
The optimal configuration produces consistent, statistically significant convergence
improvement over Kaiming init across all runs.

| Config | Delta | p-value | Significance |
|--------|-------|---------|--------------|
| n_clusters=16, n_patches=25, scale=0.2 | +0.0147 | 0.0058 | *** |
| n_clusters=16, n_patches=25, scale=0.1 | +0.0136 | 0.0119 | ** |
| n_clusters=16, n_patches=25, scale=0.25 | +0.0108 | 0.0078 | *** |

### 2. BatchNorm does not suppress the effect
The technique works within standard architectures with BatchNorm intact. The earlier
hypothesis that BatchNorm would normalize away the init signal was not supported.

### 3. n_patches is governed by foreground density
The effect is acutely sensitive to n_patches. A hard threshold exists — too few patches
yields unreliable centroids, too many floods k-means with background and dilutes the
signal. The optimal value is not fixed but is predictable from dataset foreground density.

| n_patches | Significance |
|-----------|-------------|
| 10 | ns |
| 15 | ns |
| 20 | ns |
| **25** | **\*\* to \*\*\*** |
| 30 | ns |
| 50 | ns |
| 100 | ns |

### 4. Otsu's method is the correct estimator for foreground density
Three estimators of foreground density `d` were evaluated:

- **A — Raw threshold (pixels > 0.05):** predicted n_patches=17, not significant
- **B — Batch-aware normalized (|x_norm| > 0.5):** invalid — normalization shifts
  background pixels away from zero, making this estimator semantically incorrect
- **C — Otsu's method on raw [0,1] pixels:** predicted n_patches=22, p=0.0025 (***)

Otsu's method correctly identifies the foreground/background boundary because it
optimizes the inter-class variance of the raw pixel histogram — exactly the semantic
definition of foreground density the formula requires. Estimator B is unsuitable for
normalized data and should not be used.

### 5. n_clusters = filters / 2
n_clusters=16 (half of conv1's 32 filters) consistently outperforms n_clusters=8, 32,
and 64. Conditioning exactly half the filter bank balances data-driven structure with
random diversity — conditioned filters encode dominant patterns, random filters preserve
representational breadth.

### 6. patch_size must match kernel_size
patch_size=3 dominates patch_size=5 in every comparison. A 3×3 patch maps directly onto
a 3×3 filter with no transformation, preserving full spatial structure. Larger patches
require center-cropping which discards peripheral edge information — the most
geometrically informative part of the patch.

### 7. scale is not sensitive
All scale values in [0.1, 0.3] produce significant results. Scale controls the std of
the conditioned filter weights. Kaiming produces std ≈ 0.47, so scale=0.2 is a
conservative nudge rather than a replacement. Recommended default: **scale=0.2**.

## Design Rules
Given first layer filter count `F`, kernel size `K`, and dataset foreground density `d`
estimated via Otsu's method:

| Parameter | Rule | Rationale |
|-----------|------|-----------|
| patch_size | = K | Direct geometric correspondence to filter shape |
| n_clusters | = F / 2 | Half conditioned, half random |
| n_patches | = int((F / 4) × (1 / d)) | Informative patch dominance threshold |
| scale | 0.1 – 0.3, default 0.2 | Conservative nudge, empirically insensitive |

**Reference implementation:**
```python
from PIL import Image
import numpy as np

def otsu_density(x_raw):
    """x_raw: numpy array of raw [0,1] pixel values"""
    hist, _ = np.histogram(x_raw.flatten(), bins=256, range=(0,1))
    hist     = hist.astype(float) / hist.sum()
    best_thresh, best_var = 0, 0
    for t in range(1, 256):
        w0, w1 = hist[:t].sum(), hist[t:].sum()
        if w0 == 0 or w1 == 0: continue
        m0 = (hist[:t] * np.arange(t)).sum() / w0
        m1 = (hist[t:] * np.arange(t, 256)).sum() / w1
        var = w0 * w1 * (m0 - m1) ** 2
        if var > best_var:
            best_var, best_thresh = var, t
    return (x_raw > best_thresh / 255.0).mean()

F = 32   # conv1 output channels
K = 3    # kernel size
d = otsu_density(x_raw_sample)

n_clusters = F // 2
n_patches  = int((F / 4) * (1 / d))
patch_size = K
scale      = 0.2
```

**Validation — Fashion-MNIST (F=32, K=3):**
- Otsu d = 0.3588 → predicted n_patches = 22 → observed optimum = 22 ✓ (p=0.0025)

## Limitations & Next Steps
- Results are on Fashion-MNIST only — replication on CIFAR-10 required before
  generalizing design rules
- Effect size (~0.013) is small relative to baseline variance (±0.079) — significance
  is robust at 8 seeds but magnitude is modest
- Stress test across artificially induced density conditions is ongoing — Otsu tracking
  needs to hold across all 5 threshold conditions to confirm formula generalizability
- Only the first convolutional layer is conditioned — multi-layer extension unexplored
- Formula assumes a single dominant foreground/background split — may not hold for
  datasets without clear bimodal pixel distributions (e.g., textures, medical imaging)